In [ ]:
!pip install -q easyocr pymupdf opencv-python pillow pandas openpyxl

import os
import cv2
import re
import fitz
import time
import zipfile
import shutil
import easyocr
import numpy as np
import pandas as pd
import json

from collections import defaultdict
from google.colab import files

# Total process timer (whole extraction run, in seconds)
process_start_time = time.time()

# ===========================================
# STEP 3
# Load EasyOCR
# ===========================================

reader = easyocr.Reader(
    ['en'],
    gpu=False
)

# ===========================================
# STEP 4
# Upload ZIP File
# ===========================================

uploaded = files.upload()

# Get uploaded ZIP file name
zip_file = list(uploaded.keys())[0]

# ==========================================================
# STEP 5
# Extract ZIP & Find Invoice Files
# ==========================================================

extract_folder = "/content/invoices"

# Delete old folder if it exists
if os.path.exists(extract_folder):
    shutil.rmtree(extract_folder)

# Create new folder
os.makedirs(extract_folder, exist_ok=True)

# Extract ZIP
with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

# ----------------------------------------------------------
# Find all supported invoice files
# ----------------------------------------------------------

invoice_files = []

supported_formats = (
    ".pdf",
    ".png",
    ".jpg",
    ".jpeg",
    ".tif",
    ".tiff",
    ".bmp"
)

for root, dirs, files_in_dir in os.walk(extract_folder):

    for file in files_in_dir:

        if file.lower().endswith(supported_formats):

            invoice_files.append(os.path.join(root, file))

# Sort file names
invoice_files.sort()

# ==========================================================
# STEP 6
# Convert PDF/Image into OpenCV Images
# ==========================================================

all_pages = []

for file in invoice_files:

    # ----------------------------
    # PDF
    # ----------------------------

    if file.lower().endswith(".pdf"):

        pdf = fitz.open(file)

        # Only take the first page of the PDF
        for page_no in range(min(1, len(pdf))):

            page = pdf.load_page(page_no)

            # Faster rendering (2x instead of 3x)
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))

            img = np.frombuffer(
                pix.samples,
                dtype=np.uint8
            )

            img = img.reshape(
                pix.height,
                pix.width,
                pix.n
            )

            if pix.n == 4:
                img = cv2.cvtColor(
                    img,
                    cv2.COLOR_RGBA2BGR
                )
            else:
                img = cv2.cvtColor(
                    img,
                    cv2.COLOR_RGB2BGR
                )

            all_pages.append({

                "file_name": os.path.basename(file),

                "page_no": page_no + 1,

                "image": img

            })

        pdf.close()

    # ----------------------------
    # Images
    # ----------------------------

    else:

        img = cv2.imread(file)

        all_pages.append({

            "file_name": os.path.basename(file),

            "page_no": 1,

            "image": img

        })

# ==========================================================
# STEP 7
# OCR + OCR Accuracy
# ==========================================================

invoice_data = defaultdict(lambda: {
    "rows": [],
    "confidence": [],
    "text": []
})

start = time.time()

for i, page in enumerate(all_pages):

    image = page["image"]

    # Resize large pages (improves speed)
    h, w = image.shape[:2]

    if w > 1800:

        scale = 1800 / w

        image = cv2.resize(
            image,
            None,
            fx=scale,
            fy=scale,
            interpolation=cv2.INTER_AREA
        )

    result = reader.readtext(
        image,
        detail=1,
        paragraph=False
    )

    for box, text, conf in result:

        invoice_data[page["file_name"]]["rows"].append({

            "page": page["page_no"],

            "bbox": box,

            "text": text.strip(),

            "confidence": float(conf)

        })

        invoice_data[page["file_name"]]["confidence"].append(float(conf))

        invoice_data[page["file_name"]]["text"].append(text.strip())

ocr_results = []

for file_name, data in invoice_data.items():

    if len(data["confidence"]) > 0:

        accuracy = round(
            sum(data["confidence"]) /
            len(data["confidence"])*100,
            2
        )

    else:

        accuracy = 0

    full_text = "\n".join(data["text"])

    ocr_results.append({

        "file_name": file_name,

        "ocr_accuracy": accuracy,

        "rows": data["rows"],

        "text": full_text

    })

end = time.time()

# ==========================================================
# Helper: normalize any extracted date string to YYYY/MM/DD
# ==========================================================

import datetime

def normalize_date(date_str):

    if not date_str or date_str == 0:
        return 0

    cleaned = date_str.replace("-", "/")

    possible_formats = [
        "%d/%m/%Y",
        "%d/%m/%y",
        "%Y/%m/%d",
        "%m/%d/%Y",
        "%m/%d/%y"
    ]

    for fmt in possible_formats:

        try:
            parsed = datetime.datetime.strptime(cleaned, fmt)
            return parsed.strftime("%Y/%m/%d")

        except ValueError:
            continue

    return date_str

# ==========================================================
# STEP 8
# Extract Invoice Details
# ==========================================================

invoice_results = []

for doc in ocr_results:

    # Build full OCR text
    text = "\n".join([row["text"] for row in doc["rows"]])

    invoice = {
        "File Name": doc["file_name"],
        "OCR Accuracy": doc["ocr_accuracy"],
        "Reference Number": 0,
        "Ref Date": 0,
        "E-Way Bill No": 0
    }

    # ---------------- Reference Number ----------------

    patterns = [
        r'Invoice\s*No\.?\s*[:\-]?\s*([A-Za-z0-9\/\-]+)',
        r'Invoice\s*Number\s*[:\-]?\s*([A-Za-z0-9\/\-]+)',
        r'Inv\.?\s*No\.?\s*[:\-]?\s*([A-Za-z0-9\/\-]+)',
        r'Bill\s*No\.?\s*[:\-]?\s*([A-Za-z0-9\/\-]+)',
        r'Tax\s*Invoice\s*No\.?\s*[:\-]?\s*([A-Za-z0-9\/\-]+)'
    ]

    for p in patterns:

        m = re.search(p, text, re.IGNORECASE)

        if m:
            invoice["Reference Number"] = m.group(1)
            break

    # ---------------- Ref Date ----------------

    date_patterns = [

        r'Invoice\s*Date\s*[:\-]?\s*([0-9]{2}[/-][0-9]{2}[/-][0-9]{2,4})',

        r'Date\s*[:\-]?\s*([0-9]{2}[/-][0-9]{2}[/-][0-9]{2,4})',

        r'([0-9]{2}[/-][0-9]{2}[/-][0-9]{4})'
    ]

    for p in date_patterns:

        m = re.search(p, text, re.IGNORECASE)

        if m:
            invoice["Ref Date"] = m.group(1)
            break

    invoice["Ref Date"] = normalize_date(invoice["Ref Date"])

    # ---------------- E-Way Bill ----------------

    eway_patterns = [

        r'E[- ]?Way\s*Bill\s*No\.?\s*[:\-]?\s*([0-9]{10,15})',

        r'EWB\s*No\.?\s*[:\-]?\s*([0-9]{10,15})'
    ]

    for p in eway_patterns:

        m = re.search(p, text, re.IGNORECASE)

        if m:
            invoice["E-Way Bill No"] = m.group(1)
            break

    invoice_results.append(invoice)

# ==========================================================
# STEP 9A
# Detect Table Rows
# ==========================================================

table_results = []

for doc in ocr_results:

    # Group OCR words into rows using Y-coordinate
    grouped = defaultdict(list)

    for word in doc["rows"]:

        y = int(min(pt[1] for pt in word["bbox"]) / 12)

        grouped[y].append(word)

    table_rows = []

    for key in sorted(grouped.keys()):

        row = sorted(
            grouped[key],
            key=lambda x: min(pt[0] for pt in x["bbox"])
        )

        texts = [w["text"] for w in row]

        table_rows.append(texts)

    # Detect table header
    start = -1

    header_keywords = [
        "description",
        "particular",
        "item",
        "product",
        "goods"
    ]

    for i, row in enumerate(table_rows):

        txt = " ".join(row).lower()

        if any(h in txt for h in header_keywords):

            start = i + 1
            break

    if start == -1:

        table_results.append({
            "file_name": doc["file_name"],
            "rows": [],
            "cgst_amount": 0,
            "sgst_amount": 0
        })

        continue

    # ---------------- CGST / SGST extraction ----------------

    cgst_amount = 0
    sgst_amount = 0

    for row in table_rows:

        line = " ".join(row)

        low = line.lower()

        nums = re.findall(r"\d[\d,]*\.?\d*", line)

        if "cgst" in low and nums:
            cgst_amount = nums[-1].replace(",", "")

        elif "sgst" in low and nums:
            sgst_amount = nums[-1].replace(",", "")

    extracted_rows = []

    for row in table_rows[start:]:

        line = " ".join(row)

        low = line.lower()

        if any(x in low for x in [
            "grand total",
            "total",
            "cgst",
            "sgst",
            "igst",
            "round off",
            "amount chargeable",
            "terms",
            "bank",
            "authorized"
        ]):
            break

        extracted_rows.append(row)

    table_results.append({

        "file_name": doc["file_name"],

        "rows": extracted_rows,

        "cgst_amount": cgst_amount,

        "sgst_amount": sgst_amount

    })

# ==========================================================
# STEP 9B
# Extract Line Items
# ==========================================================

line_items = []

for tbl in table_results:

    invoice_items = []

    for row in tbl["rows"]:

        line = " ".join(row)

        if len(line.strip()) < 5:
            continue

        numbers = re.findall(r"\d[\d,]*\.?\d*", line)

        if len(numbers) < 3:
            continue

        amount = numbers[-1].replace(",", "")

        rate = numbers[-2].replace(",", "")

        qty = numbers[-3].replace(",", "")

        # Tax Rate (per item GST/CGST+SGST %, e.g. "18%")

        tax_rate = 0

        m = re.search(r"\b(\d{1,2}(?:\.\d+)?)\s*%", line)

        if m:
            tax_rate = m.group(1)

        # Unit

        unit = 0

        m = re.search(
            r"\b(NOS|NO|PCS|PC|UNIT|KG|KGS|LTR|LTRS|BOX|BAG|ROLL|SET|MTR|MT)\b",
            line,
            re.I
        )

        if m:
            unit = m.group().upper()

        # HSN

        hsn = 0

        m = re.search(r"\b\d{4,8}\b", line)

        if m:
            hsn = m.group()

        # Description

        desc = line

        desc = re.sub(r"\b\d{1,2}(?:\.\d+)?\s*%", "", desc)

        for n in numbers:
            desc = desc.replace(n, "")

        desc = re.sub(
            r"\b(NOS|NO|PCS|PC|UNIT|KG|KGS|LTR|LTRS|BOX|BAG|ROLL|SET|MTR|MT)\b",
            "",
            desc,
            flags=re.I
        )

        desc = re.sub(r"\|", " ", desc)

        desc = " ".join(desc.split())

        invoice_items.append({

            "itemdesc": desc,

            "hsncode": hsn,

            "qty": qty,

            "unit": unit,

            "rate": rate,

            "amount": amount,

            "taxper": tax_rate

        })

    line_items.append({

        "file_name": tbl["file_name"],

        "items": invoice_items

    })

# ==========================================================
# STEP 10
# FINAL REPORT (per file: fields + item lines + JSON, then next file)
# ==========================================================

# Create lookup for line items
line_item_dict = {}

for item in line_items:
    line_item_dict[item["file_name"]] = item["items"]

# Create lookup for cgst/sgst per file
tax_dict = {}

for tbl in table_results:
    tax_dict[tbl["file_name"]] = {
        "cgst_amount": tbl.get("cgst_amount", 0),
        "sgst_amount": tbl.get("sgst_amount", 0)
    }

final_output = []

for invoice in invoice_results:

    file_name = invoice["File Name"]

    items = line_item_dict.get(file_name, [])

    tax = tax_dict.get(file_name, {"cgst_amount": 0, "sgst_amount": 0})

    print("\n" + "="*100)
    print("File Name       :", file_name)
    print("Reference Number:", invoice["Reference Number"])
    print("Ref Date        :", invoice["Ref Date"])
    print("OCR Accuracy    :", invoice["OCR Accuracy"], "%")
    print("-"*100)

    if len(items) == 0:

        print("No Line Items Found")

    else:

        for itm in items:

            print(
                f"{itm['itemdesc']} | "
                f"HSN: {itm['hsncode']} | Qty: {itm['qty']} | "
                f"Unit: {itm['unit']} | Rate: {itm['rate']} | Amount: {itm['amount']} | "
                f"Tax: {itm['taxper']}%"
            )

    print("CGST Amount     :", tax["cgst_amount"])
    print("SGST Amount     :", tax["sgst_amount"])

    file_json = {

        "filename": file_name,

        "ocraccuracy": invoice["OCR Accuracy"],

        "refno": invoice["Reference Number"],

        "refdate": invoice["Ref Date"],

        "cgstamount": tax["cgst_amount"],

        "sgstamount": tax["sgst_amount"],

        "items": items

    }

    print("\nJSON :")
    print(json.dumps(file_json, indent=4))

    final_output.append(file_json)

process_end_time = time.time()

print("\n" + "="*100)
print("Total Extraction Time :", round(process_end_time - process_start_time, 2), "Seconds")
print("PROCESS COMPLETED")
print("="*100)